<a href="https://colab.research.google.com/github/sanil-edwin/llm-agents/blob/main/L10_agents_travel_planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="padding-top: 25px;padding-bottom: 25px;text-align: left; padding-left: 10px; background-color: #DDDDDD;
    color: black;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png" height="50px"> <a href='https://harvard-iacs.github.io/2025-AC215/' target='_blank'><strong><font color="#A41034">AC215/E115: MLOps & LLMOps: Production AI Systems</font></strong></a></h1>

# **<font color="#A41034">Tutorial -  Agents - Travel Planner</font>**

**Harvard University**<br/>
**Fall 2025**<br/>
**Instructor:** Pavlos Protopapas<br/>


<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Working with Agents**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



In this notebook, we will build an LLM agent from scratch. We will build a LLM based Travel Agent using mock tools to make searching hotels, flights, restaurants simpler. In a real LLM based Travel Planner you would connect to real APIs for Hotels, Airlines, Restaurant searches.

# **Learning Objectives**

By the end of this tutorial/guided demo, you will be able to:

- **Understand AI Agents:**
  - Define agents and their role in workflows.
  - Explain agentic workflows for complex tasks.

- **Set Up Mock Tools:**
  - Set up mock tools for City, Hotel, Flight, Restaurant, Attractions search

- **Implement LLM AI Agents:**
  - Implement AI agent for Travel planning.
  - Integrate Agents with the mock tools.

- **Analyze Applications:**
  - Review and analyze real-world travel planning needs.


# **Understanding Agents**

**What are Agents?**<br>
---
Agent is a component that uses an LLM with instructions and tools, that can run autonomously to accomplish tasks.

An agentic workflow is any multi-step process that iteratively instructs large language models to complete complex tasks. <br>

So, we can think of agents here as a part of an agentic workflow that completes a particular task and hands over the result to the next agent, all of them in tandem utilising LLMs to complete the larger task given as a prompt.

# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [ ]:
# @title Imports
import os
import pandas as pd
import re
import glob
import json
from typing import Any
import random
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Optional, Union, Set
from datetime import datetime, timedelta
import shutil
import traceback
import uuid
from typing import Dict, List, Any, Optional, Callable
from openai import OpenAI

In [ ]:
# @title Setup OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

# **Prompt Templates**

In this section we will create all the prompts we will be using in this notebook to implement the AI Agent

In [ ]:
# @title Query Understanding Prompt

# Prompts
QUERY_UNDERSTANDING_TMPL = """
You are a travel query parser that converts natural language queries into structured format.
Follow these steps carefully:

1. UNDERSTANDING PHASE
Analyze the following travel query and identify key components:
{user_query}

2. EXTRACTION RULES
- Dates: Extract all dates in ISO format (YYYY-MM-DD)
- Locations: Identify source and destination locations
- Budget: Extract numerical budget values and currency
- Duration: Calculate trip duration from dates
- Preferences: Process special requirements

3. ACCOMMODATION RULES
Accommodation type must be one of these exact values:
- "entire_room" when "entire room" is mentioned
- "hotel" when "hotel" or "private room" is mentioned
- "apartment" when "apartment" is mentioned
- Default to "hotel" if no type specified

Requirements must be in this exact format:
- Use "pet_friendly" (not "pet-friendly" or "pet friendly")

Provide your response in valid JSON format with preferences formatted exactly as specified above.

## Output Schema
```json
{
    "query_type": "travel_planning",
    "temporal": {
        "start_date": "YYYY-MM-DD",
        "end_date": "YYYY-MM-DD",
        "duration_days": 0
    },
    "spatial": {
        "origin": "",
        "destinations": [],
        "route_type": "one_way"
    },
    "budget": {
        "amount": 0,
        "currency": "USD",
        "constraints": []
    },
    "preferences": {
        "accommodation": {
            "type": "",
            "requirements": []
        },
        "transportation": {
            "preferred_modes": []
        },
        "activities": []
    },
    "constraints": {
        "hard": [],
        "soft": []
    }
}
```
"""

In [ ]:
# @title Agentic Core Prompt

AGENTIC_CORE_TMPL = """
You are a travel planning assistant. Given the user query:
1. Analyze the requirements
2. Create a step-by-step plan
3. Execute the plan using available tools
4. Ask for user input when needed
5. Provide a final response when complete

Do not repond to any questions that is not about travel planning.

Available tools will be provided in the function schema.
"""

# **Define Tools**

In this section we will create mock tools that will be used by the AI Agent for Travel planning

In [ ]:
# @title Query Understanding

class QueryUnderstanding:
    def __init__(self):
        self.template = QUERY_UNDERSTANDING_TMPL

    def execute(self, user_query: str) -> Dict:
        """Parse a natural language travel query into structured JSON format."""
        try:
            # Initialize OpenAI client
            client = OpenAI()

            # Format the prompt with user query
            input_prompt = QUERY_UNDERSTANDING_TMPL.replace("{user_query}", user_query)

            # Call OpenAI API
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "You are a travel query parsing assistant. Provide responses in valid JSON format only."},
                    {"role": "user", "content": input_prompt}
                ],
                temperature=0.0,  # Low temperature for consistent, structured output
                response_format={"type": "json_object"}  # Ensure JSON response
            )

            # Extract and parse JSON response
            response_text = response.choices[0].message.content
            parsed_response = json.loads(response_text)

            # Basic validation of required fields
            required_fields = ["query_type", "temporal", "spatial", "budget"]
            for field in required_fields:
                if field not in parsed_response:
                    raise ValueError(f"Missing required field: {field}")

            return parsed_response

        except json.JSONDecodeError as e:
            print(f"Error parsing JSON response: {e}")
            traceback.print_exc()
            return None
        except Exception as e:
            print(f"Error processing query: {e}")
            traceback.print_exc()
            return None


In [ ]:
# @title City Search

class CitySearchTool:
    def __init__(self):
        # Mock city database
        self.cities = {
            "Seattle": {
                "population": 737015,
                "attractions": ["Space Needle", "Pike Place Market", "Museum of Pop Culture"],
                "airport": "SEA"
            },
            "San Francisco": {
                "population": 873965,
                "attractions": ["Golden Gate Bridge", "Alcatraz", "Fisherman's Wharf"],
                "airport": "SFO"
            },
            "Los Angeles": {
                "population": 3967000,
                "attractions": ["Hollywood Sign", "Universal Studios", "Santa Monica Pier"],
                "airport": "LAX"
            }
        }

    def execute(self, city_name: str) -> Optional[dict]:
        """Search for city information."""
        if city_name not in self.cities:
            return None

        city_data = self.cities[city_name]
        return {
            "city": city_name,
            "population": city_data["population"],
            "attractions": city_data["attractions"]
        }

In [ ]:
# @title Flight Search

class FlightSearchTool():
    def __init__(self):
        # Simulated flight data
        self.airlines = ["AA:American Airlines", "DA:Delta", "JB:Jet Blue", "SW:Southwest"]
        self.base_prices = {
            ("Seattle", "San Francisco"): 250,
            ("San Francisco", "Seattle"): 240,
            ("Seattle", "Los Angeles"): 300,
            ("San Francisco", "Los Angeles"): 200,
            ("Los Angeles", "Las Vegas"): 150,
            ("San Francisco", "Las Vegas"): 200,
        }

    def execute(self,
                departure_city: str,
                arrival_city: str,
                date: str,
                class_type: str = "economy") -> List[dict]:
        """Search for flights between cities."""
        results = []
        base_price = self.base_prices.get(
            (departure_city, arrival_city),
            random.uniform(200, 500)
        )

        # Generate 3-5 flight options
        num_flights = random.randint(3, 5)

        # Convert date string to datetime object
        # Expected format: "YYYY-MM-DD"
        flight_date = datetime.strptime(date, "%Y-%m-%d")

        for _ in range(num_flights):
            # Randomize departure times throughout the day
            departure_hour = random.randint(6, 20)
            departure_time = flight_date.replace(hour=departure_hour, minute=random.randint(0, 55))

            # Calculate flight duration based on distance (simplified)
            duration = timedelta(hours=random.uniform(1, 5))

            # Add some price variation
            price_variation = random.uniform(0.8, 1.4)
            final_price = base_price * price_variation

            # Adjust price for class type
            if class_type.lower() == "business":
                final_price *= 2.5
            elif class_type.lower() == "first":
                final_price *= 4

            # Convert datetime to string
            arrival_time = departure_time + duration
            departure_str = departure_time.strftime("%Y-%m-%d %H:%M")
            arrival_str = arrival_time.strftime("%Y-%m-%d %H:%M")

            results.append({
                "flight_number": f"{random.choice(self.airlines)[:2]}{random.randint(100, 999)}",
                "airline": random.choice(self.airlines),
                "departure_city": departure_city,
                "arrival_city": arrival_city,
                "departure_time": departure_str,
                "arrival_time": arrival_str,
                "price": round(final_price, 2),
                "seats_available": random.randint(0, 50),
                "class_type": class_type
            })

        return sorted(results, key=lambda x: x["price"])


In [ ]:
# @title Hotel Search

class HotelSearchTool:
    def __init__(self):
        self.hotels = {
            "Seattle": [
                {"name": "Seattle Downtown Hotel", "base_price": 200, "rating": 4.5, "pet_friendly": True},
                {"name": "Space View Inn", "base_price": 150, "rating": 4.0, "pet_friendly": False},
                {"name": "Harbor Lodge", "base_price": 300, "rating": 4.8, "pet_friendly": True}
            ],
            "San Francisco": [
                {"name": "Bay Area Suite", "base_price": 500, "rating": 4.6, "pet_friendly": True},
                {"name": "Golden Gate Inn", "base_price": 450, "rating": 4.2, "pet_friendly": False}
            ],
            "Los Angeles": [
                {"name": "LA Downtown Hotel", "base_price": 280, "rating": 4.4, "pet_friendly": True},
                {"name": "Hollywood Stars Hotel", "base_price": 220, "rating": 4.1, "pet_friendly": False}
            ]
        }
        self.amenities = ["WiFi", "Pool", "Gym", "Restaurant", "Bar", "Spa", "Parking"]

    def execute(self, city: str, check_in: str, check_out: str,
                room_type: str = "standard", pet_friendly: bool = False) -> List[dict]:
        """Search for hotels."""
        print("Search for hotels.")
        if city not in self.hotels:
            return []

        check_in_date = datetime.strptime(check_in, "%Y-%m-%d")
        check_out_date = datetime.strptime(check_out, "%Y-%m-%d")

        results = []
        for hotel in self.hotels[city]:
            if pet_friendly and not hotel["pet_friendly"]:
                continue

            price_multiplier = 1.5 if room_type == "suite" else 1
            results.append({
                "hotel_name": hotel["name"],
                "city":city,
                "check_in": check_in,
                "check_out": check_out,
                "room_type": room_type,
                "price_per_night": hotel["base_price"] * price_multiplier,
                "rating": hotel["rating"],
                "amenities": random.sample(self.amenities, k=random.randint(3, len(self.amenities))),
                "is_pet_friendly": hotel["pet_friendly"]
            })
        return sorted(results, key=lambda x: x["rating"], reverse=True)


In [ ]:
# @title Restaurant Search

class RestaurantSearchTool():
    def __init__(self):
        self.cuisines = ["Italian", "Japanese", "Mexican", "American", "Chinese", "French"]
        self.restaurant_types = ["Casual", "Fine Dining", "Fast Casual", "Bistro"]

    def execute(self,
                city: str,
                cuisine: Optional[str] = None,
                price_range: Optional[str] = None) -> List[Dict]:
        """Search for restaurants in a city."""
        results = []

        # Generate 5-10 restaurant options
        num_restaurants = random.randint(5, 10)

        for _ in range(num_restaurants):
            selected_cuisine = cuisine or random.choice(self.cuisines)
            selected_type = random.choice(self.restaurant_types)

            # Generate price range if not specified
            if not price_range:
                price_range = random.choice(["$", "$$", "$$$", "$$$$"])

            results.append({
                "name": f"{selected_cuisine.title()} {selected_type}",
                "cuisine": selected_cuisine,
                "type": selected_type,
                "city": city,
                "price_range": price_range,
                "rating": round(random.uniform(3.5, 5.0), 1),
                "reviews_count": random.randint(50, 1000),
                "popular_dishes": [
                    f"Dish {i+1}" for i in range(random.randint(2, 5))
                ],
                "wait_time": random.randint(5, 60)
            })

        return sorted(results, key=lambda x: x["rating"], reverse=True)

In [ ]:
# @title Attraction Search

class AttractionSearchTool:
    def __init__(self):
        # Mock attraction database
        self.attractions = {
            "Seattle": [
                {
                    "name": "Space Needle",
                    "category": "Landmark",
                    "rating": 4.8,
                    "price_range": "$$"
                },
                {
                    "name": "Pike Place Market",
                    "category": "Shopping",
                    "rating": 4.6,
                    "price_range": "$"
                }
            ],
            "San Francisco": [
                {
                    "name": "Golden Gate Bridge",
                    "category": "Landmark",
                    "rating": 4.9,
                    "price_range": "$"
                },
                {
                    "name": "Fisherman's Wharf",
                    "category": "Entertainment",
                    "rating": 4.5,
                    "price_range": "$$"
                }
            ],
            "Los Angeles": [
                {
                    "name": "Universal Studios Hollywood",
                    "category": "Theme park",
                    "rating": 4.9,
                    "price_range": "$$$"
                },
                {
                    "name": "Petersen Automotive Museum",
                    "category": "Museum",
                    "rating": 4.5,
                    "price_range": "$"
                },
                {
                    "name": "Griffith Observatory",
                    "category": "Landmark",
                    "rating": 4.9,
                    "price_range": "$"
                }
            ]
        }

    def execute(self, city: str, category: Optional[str] = None) -> List[dict]:
        """Search for attractions."""
        if city not in self.attractions:
            return []

        results = []
        for attr in self.attractions[city]:
            if category and attr["category"] != category:
                continue
            results.append({
                "name": attr["name"],
                "city": city,
                "category": attr["category"],
                "rating": attr["rating"],
                "price_range": attr["price_range"],
                "description": f"Popular {attr['category'].lower()} in {city}",
                "opening_hours": {
                    "weekdays": "9:00 AM - 5:00 PM",
                    "weekends": "10:00 AM - 6:00 PM"
                },
                "popular_times": {
                    "morning": random.randint(30, 70),
                    "afternoon": random.randint(60, 90),
                    "evening": random.randint(40, 80)
                }
            })
        return sorted(results, key=lambda x: x["rating"], reverse=True)


In [ ]:
# Test
attraction_search = AttractionSearchTool()
attraction_search.execute("Los Angeles")

[{'name': 'Universal Studios Hollywood',
  'city': 'Los Angeles',
  'category': 'Theme park',
  'rating': 4.9,
  'price_range': '$$$',
  'description': 'Popular theme park in Los Angeles',
  'opening_hours': {'weekdays': '9:00 AM - 5:00 PM',
   'weekends': '10:00 AM - 6:00 PM'},
  'popular_times': {'morning': 67, 'afternoon': 84, 'evening': 72}},
 {'name': 'Griffith Observatory',
  'city': 'Los Angeles',
  'category': 'Landmark',
  'rating': 4.9,
  'price_range': '$',
  'description': 'Popular landmark in Los Angeles',
  'opening_hours': {'weekdays': '9:00 AM - 5:00 PM',
   'weekends': '10:00 AM - 6:00 PM'},
  'popular_times': {'morning': 43, 'afternoon': 86, 'evening': 74}},
 {'name': 'Petersen Automotive Museum',
  'city': 'Los Angeles',
  'category': 'Museum',
  'rating': 4.5,
  'price_range': '$',
  'description': 'Popular museum in Los Angeles',
  'opening_hours': {'weekdays': '9:00 AM - 5:00 PM',
   'weekends': '10:00 AM - 6:00 PM'},
  'popular_times': {'morning': 37, 'afternoon'

# **Setup Agent**

In this section we setup all the components we need to run an AI Agent for Travel planning

In [ ]:
# @title Agent Tools Config

AGENT_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "query_understanding",
            "description": "Convert natural language travel query into structured format with temporal, spatial, budget, and preference information",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_query": {
                        "type": "string",
                        "description": "Natural language travel query from user"
                    }
                },
                "required": ["user_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "city_search",
            "description": "Search for information about a city including population and attractions",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "Name of the city to search for"
                    }
                },
                "required": ["city_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "flight_search",
            "description": "Search for flights between cities on specific dates",
            "parameters": {
                "type": "object",
                "properties": {
                    "departure_city": {
                        "type": "string",
                        "description": "City of departure"
                    },
                    "arrival_city": {
                        "type": "string",
                        "description": "City of arrival"
                    },
                    "date": {
                        "type": "string",
                        "description": "Date of travel in ISO format (YYYY-MM-DD)"
                    },
                    "class_type": {
                        "type": "string",
                        "description": "Class of travel",
                        "enum": ["economy", "business", "first"],
                        "default": "economy"
                    }
                },
                "required": ["departure_city", "arrival_city", "date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "hotel_search",
            "description": "Search for hotels in a city with specific criteria",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City to search in"
                    },
                    "check_in": {
                        "type": "string",
                        "description": "Check-in date in ISO format (YYYY-MM-DD)"
                    },
                    "check_out": {
                        "type": "string",
                        "description": "Check-out date in ISO format (YYYY-MM-DD)"
                    },
                    "room_type": {
                        "type": "string",
                        "description": "Type of room",
                        "enum": ["standard", "suite", "entire_room"],
                        "default": "standard"
                    },
                    "pet_friendly": {
                        "type": "boolean",
                        "description": "Whether the hotel needs to be pet-friendly",
                        "default": False
                    }
                },
                "required": ["city", "check_in", "check_out"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restaurant_search",
            "description": "Search for restaurants in a city with optional cuisine and price range filters",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City to search in"
                    },
                    "cuisine": {
                        "type": "string",
                        "description": "Type of cuisine (optional)",
                        "enum": ["Italian", "Japanese", "Mexican", "American", "Chinese", "French"]
                    },
                    "price_range": {
                        "type": "string",
                        "description": "Price range (optional)",
                        "enum": ["$", "$", "$$", "$$"]
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "attraction_search",
            "description": "Search for attractions in a city with optional category filter",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City to search in"
                    },
                    "category": {
                        "type": "string",
                        "description": "Category of attraction (optional)",
                        "enum": ["Landmark", "Shopping", "Entertainment", "Museum", "Park"]
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Initialize tools
query_understanding = QueryUnderstanding()
city_search = CitySearchTool()
flight_search = FlightSearchTool()
hotel_search = HotelSearchTool()
restaurant_search = RestaurantSearchTool()
attraction_search = AttractionSearchTool()

# Tools dictionary mapping function names to actual implementations
AGENT_TOOLS = {
    "query_understanding": query_understanding.execute,
    "city_search": city_search.execute,
    "flight_search": flight_search.execute,
    "hotel_search": hotel_search.execute,
    "restaurant_search": restaurant_search.execute,
    "attraction_search": attraction_search.execute
}


In [ ]:
# @title Multi Agent System: Travel Agent

# LLM Model
OPENAI_MODEL = "gpt-4o-mini"

class PlanStatus(Enum):
    """Status of the current plan execution"""
    NOT_STARTED = auto()
    IN_PROGRESS = auto()
    WAITING_USER_INPUT = auto()
    COMPLETED = auto()
    FAILED = auto()

@dataclass
class ToolResult:
    """Results from tool execution"""
    tool_name: str
    success: bool
    result: Any
    error_message: Optional[str] = None

@dataclass
class ExecutionState:
    """Tracks the current state of plan execution"""
    status: PlanStatus
    current_step: int
    total_steps: int
    tool_results: List[ToolResult]
    conversation_history: List[Dict]

class TravelAgent:
    """
    Central decision-making component that coordinates between other modules
    and maintains the overall state of the travel planning process.
    """

    def __init__(self):
        self.model = OPENAI_MODEL
        self.client = OpenAI()
        self.state = ExecutionState(
            status=PlanStatus.NOT_STARTED,
            current_step=0,
            total_steps=0,
            tool_results=[],
            conversation_history=[]
        )

        # Register tools with function calling schema
        self.tool_schemas = AGENT_TOOLS_SCHEMA
        self.tools = AGENT_TOOLS

    def _generate_tool_schemas(self) -> List[Dict]:
        """Generate OpenAI function schemas for registered tools"""
        # This would parse function signatures and docstrings to create OpenAI function schemas
        # For now returning a placeholder - we'll implement this next
        schemas = []
        return schemas

    def _update_conversation_history(self, role: str, content: str):
        """Add new message to conversation history"""
        self.state.conversation_history.append({
            "role": role,
            "content": content
        })

    def _execute_tool(self, tool_name: str, **kwargs) -> ToolResult:
        """Execute a specific tool with given parameters"""
        try:
            if tool_name not in self.tools:
                raise ValueError(f"Tool {tool_name} not found")

            print("Executing Tool:", tool_name, "Arguments:", kwargs)
            result = self.tools[tool_name](**kwargs)
            print("Tool Results:", result)
            return ToolResult(
                tool_name=tool_name,
                success=True,
                result=result
            )
        except Exception as e:
            print(f"Error executing tool: {e}")
            traceback.print_exc()
            return ToolResult(
                tool_name=tool_name,
                success=False,
                result=None,
                error_message=str(e)
            )

    def _get_llm_response(self) -> Dict:
        """Get response from LLM including potential tool calls"""
        try:
            messages = [{"role": "system", "content": AGENTIC_CORE_TMPL}]
            messages.extend(self.state.conversation_history)

            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=self.tool_schemas
            )

            return response.choices[0].message
        except Exception as e:
            print(f"Error getting LLM response: {e}")
            return None

    def handle_user_query(self, query: str, is_additional_info=False) -> str:
        """
        Main entry point for handling user queries.
        """
        if not is_additional_info:
            # Reset state for new query
            self.state = ExecutionState(
                status=PlanStatus.IN_PROGRESS,
                current_step=0,
                total_steps=0,
                tool_results=[],
                conversation_history=[]
            )
        else:
            self.state.status = PlanStatus.IN_PROGRESS

        # Add user query to conversation
        self._update_conversation_history("user", query)

        while self.state.status in [PlanStatus.IN_PROGRESS]:
            # Get next action from LLM
            llm_response = self._get_llm_response()
            #print("llm_response:\n", llm_response)

            if not llm_response:
                self.state.status = PlanStatus.FAILED
                return "I apologize, but I encountered an error processing your request."

            # Handle different types of LLM responses
            if hasattr(llm_response, 'tool_calls') and llm_response.tool_calls:
                # Execute tool calls
                for tool_call in llm_response.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)

                    result = self._execute_tool(tool_name, **tool_args)
                    self.state.tool_results.append(result)

                    # Add tool result to conversation
                    self._update_conversation_history(
                        "assistant",
                        f"Tool {tool_name} returned: {json.dumps(result.result)}"
                    )
            else:
                # Message is for user
                self._update_conversation_history("assistant", llm_response.content)
                self.state.status = PlanStatus.WAITING_USER_INPUT
                return llm_response.content

        # Something went wrong with the processing
        return "I apologize, but I was unable to complete the travel planning process."

# **Travel Planner Agent in Action**

## Example 1:

User Query:

 *I'm going from Seattle to California from November 6 to 10, 2025. I have a budget of $6000. For lodging, I prefer an entire room and the accommodations must be pet-friendly*




In [ ]:
# Initialize Agent
agent_1 = TravelAgent()

# User Query
user_query = """
I'm going from Seattle to California from November 6 to 10, 2025.
I have a budget of $6000. For lodging, I prefer an entire room
and the accommodations must be pet-friendly
"""

print("=== Travel Planning Agent Test ===")
print("\nOriginal Query:", user_query)
print("\n=== Starting Query Processing ===")

try:
    # Process the initial query
    response = agent_1.handle_user_query(user_query)
    print("response:", response)

    print("\n***********\n")

    # Add San Francisco
    print("Adding user input:","San Francisco")
    response = agent_1.handle_user_query("San Francisco", is_additional_info=True)
    print("response:", response)

    print("\n\n\n=== Conversation History ===")
    for message in agent_1.state.conversation_history:
        print(f"\n{message['role'].upper()}:")
        print(message['content'])

    print("\n=== Tool Execution Results ===")
    for result in agent_1.state.tool_results:
        print(f"\nTool: {result.tool_name}")
        print(f"Success: {result.success}")
        if result.success:
            print("Result:", json.dumps(result.result, indent=2))
        else:
            print("Error:", result.error_message)

    print("\n=== Final Response ===")
    print(response)

    print("\n=== Final State ===")
    print(f"Status: {agent_1.state.status}")
    print(f"Steps Completed: {agent_1.state.current_step} of {agent_1.state.total_steps}")

except Exception as e:
    print(f"\nError occurred during processing: {str(e)}")
    traceback.print_exc()

=== Travel Planning Agent Test ===

Original Query: 
I'm going from Seattle to California from March 6 to 10, 2025.
I have a budget of $6000. For lodging, I prefer an entire room
and the accommodations must be pet-friendly


=== Starting Query Processing ===
Executing Tool: query_understanding Arguments: {'user_query': "I'm going from Seattle to California from March 6 to 10, 2025. I have a budget of $6000. For lodging, I prefer an entire room and the accommodations must be pet-friendly."}
Tool Results: {'query_type': 'travel_planning', 'temporal': {'start_date': '2025-03-06', 'end_date': '2025-03-10', 'duration_days': 4}, 'spatial': {'origin': 'Seattle', 'destinations': ['California'], 'route_type': 'one_way'}, 'budget': {'amount': 6000, 'currency': 'USD', 'constraints': []}, 'preferences': {'accommodation': {'type': 'entire_room', 'requirements': ['pet_friendly']}, 'transportation': {'preferred_modes': []}, 'activities': []}, 'constraints': {'hard': [], 'soft': []}}
response: Here's 

In [ ]:
additional_user_input = """
Book Jet Blue Flight SW533 and Bay Area Suite. Also suggest some restaurants for dinner options for the nights of 6th, 7th, 8th
"""
response = agent_1.handle_user_query(additional_user_input, is_additional_info=True)
print("response:", response)

Executing Tool: flight_search Arguments: {'departure_city': 'Seattle', 'arrival_city': 'San Francisco', 'date': '2025-03-06', 'class_type': 'economy'}
Tool Results: [{'flight_number': 'JB195', 'airline': 'JB:Jet Blue', 'departure_city': 'Seattle', 'arrival_city': 'San Francisco', 'departure_time': '2025-03-06 07:50', 'arrival_time': '2025-03-06 11:56', 'price': 200.29, 'seats_available': 2, 'class_type': 'economy'}, {'flight_number': 'SW681', 'airline': 'DA:Delta', 'departure_city': 'Seattle', 'arrival_city': 'San Francisco', 'departure_time': '2025-03-06 19:31', 'arrival_time': '2025-03-06 20:49', 'price': 213.06, 'seats_available': 24, 'class_type': 'economy'}, {'flight_number': 'AA163', 'airline': 'JB:Jet Blue', 'departure_city': 'Seattle', 'arrival_city': 'San Francisco', 'departure_time': '2025-03-06 19:18', 'arrival_time': '2025-03-06 23:45', 'price': 221.31, 'seats_available': 31, 'class_type': 'economy'}, {'flight_number': 'SW191', 'airline': 'JB:Jet Blue', 'departure_city': 'S

In [ ]:
additional_user_input = """
Add Italian on the 6th, Mexican on the 7th, and French on the 8th. Then finalize my travel plan and create a consolidated view for my plans
"""
response = agent_1.handle_user_query(additional_user_input, is_additional_info=True)
print("response:", response)

Executing Tool: restaurant_search Arguments: {'city': 'San Francisco', 'cuisine': 'Italian', 'price_range': ''}
Tool Results: [{'name': 'Italian Fast Casual', 'cuisine': 'Italian', 'type': 'Fast Casual', 'city': 'San Francisco', 'price_range': '$', 'rating': 4.9, 'reviews_count': 65, 'popular_dishes': ['Dish 1', 'Dish 2', 'Dish 3', 'Dish 4', 'Dish 5'], 'wait_time': 12}, {'name': 'Italian Casual', 'cuisine': 'Italian', 'type': 'Casual', 'city': 'San Francisco', 'price_range': '$', 'rating': 4.7, 'reviews_count': 192, 'popular_dishes': ['Dish 1', 'Dish 2', 'Dish 3', 'Dish 4'], 'wait_time': 21}, {'name': 'Italian Fine Dining', 'cuisine': 'Italian', 'type': 'Fine Dining', 'city': 'San Francisco', 'price_range': '$', 'rating': 4.6, 'reviews_count': 388, 'popular_dishes': ['Dish 1', 'Dish 2', 'Dish 3', 'Dish 4', 'Dish 5'], 'wait_time': 11}, {'name': 'Italian Fast Casual', 'cuisine': 'Italian', 'type': 'Fast Casual', 'city': 'San Francisco', 'price_range': '$', 'rating': 4.4, 'reviews_count':

## Example 2:

User Query:

 *I'm going from Seattle to Los Angeles from December 15 to 18, 2025. My visit is primarily to visit as many attractions as possible on the 16th and 17th. My food preferences are Mexican and Italian only. I would like to stay in a hotel and my total budget is $3000.*



In [ ]:
# Initialize Agent
agent_2 = TravelAgent()

# User Query
user_query = """
I'm going from Seattle to Los Angeles from December 15 to 18, 2025. My visit is primarily to visit as many attractions
as possible on the 16th and 17th. My food preferences are Mexican and Italian only. I would like to stay in a hotel
and my total budget is $3000.
"""

print("=== Travel Planning Agent Test ===")
print("\nOriginal Query:", user_query)
print("\n=== Starting Query Processing ===")

# Process the initial query
response = agent_2.handle_user_query(user_query)
print("response:", response)

=== Travel Planning Agent Test ===

Original Query: 
I'm going from Seattle to Los Angeles from April 15 to 18, 2025. My visit is primarily to visit as many attractions
as possible on the 16th and 17th. My food preferences are Mexican and Italian only. I would like to stay in a hotel
and my total budget is $3000.


=== Starting Query Processing ===
Executing Tool: query_understanding Arguments: {'user_query': "I'm going from Seattle to Los Angeles from April 15 to 18, 2025. My visit is primarily to visit as many attractions as possible on the 16th and 17th. My food preferences are Mexican and Italian only. I would like to stay in a hotel and my total budget is $3000."}
Tool Results: {'query_type': 'travel_planning', 'temporal': {'start_date': '2025-04-15', 'end_date': '2025-04-18', 'duration_days': 3}, 'spatial': {'origin': 'Seattle', 'destinations': ['Los Angeles'], 'route_type': 'one_way'}, 'budget': {'amount': 3000, 'currency': 'USD', 'constraints': []}, 'preferences': {'accommodati

In [ ]:
additional_user_input = """
Book flight SW545 (Delta Airlines) and stay at LA Downtown Hotel. Suggest a few attactions
"""
response = agent_1.handle_user_query(additional_user_input, is_additional_info=True)
print("response:", response)

Executing Tool: flight_search Arguments: {'departure_city': 'Seattle', 'arrival_city': 'Los Angeles', 'date': '2025-03-06', 'class_type': 'economy'}
Tool Results: [{'flight_number': 'JB296', 'airline': 'SW:Southwest', 'departure_city': 'Seattle', 'arrival_city': 'Los Angeles', 'departure_time': '2025-03-06 08:05', 'arrival_time': '2025-03-06 12:15', 'price': 293.32, 'seats_available': 50, 'class_type': 'economy'}, {'flight_number': 'DA331', 'airline': 'SW:Southwest', 'departure_city': 'Seattle', 'arrival_city': 'Los Angeles', 'departure_time': '2025-03-06 08:23', 'arrival_time': '2025-03-06 12:55', 'price': 384.04, 'seats_available': 14, 'class_type': 'economy'}, {'flight_number': 'AA886', 'airline': 'JB:Jet Blue', 'departure_city': 'Seattle', 'arrival_city': 'Los Angeles', 'departure_time': '2025-03-06 09:46', 'arrival_time': '2025-03-06 14:06', 'price': 386.31, 'seats_available': 27, 'class_type': 'economy'}, {'flight_number': 'AA727', 'airline': 'DA:Delta', 'departure_city': 'Seattl

## Example 3:

In [ ]:
# Initialize Agent
agent_3 = TravelAgent()

# User Query
user_query = """
I am doing a workshop and need help preparing slides about LLMs. Can you come up with a plan for me?
"""

print("=== Travel Planning Agent Test ===")
print("\nOriginal Query:", user_query)
print("\n=== Starting Query Processing ===")

# Process the initial query
response = agent_3.handle_user_query(user_query)
print("response:", response)

=== Travel Planning Agent Test ===

Original Query: 
I am doing a workshop and need help preparing slides about LLMs. Can you come up with a plan for me?


=== Starting Query Processing ===
response: I'm here to assist with travel planning only. If you have any travel-related queries or needs, feel free to let me know!
